# Cargills Data Engineering Assessment
## Bronze Ingestion

### Purpose
This notebook processes raw sales batch files that have been
landed in the Bronze raw storage area by the master Fabric pipeline.

### Responsibilities
- Read the raw CSV batch from the Lakehouse Bronze landing zone.
- Preserve source traceability through ingestion metadata.
- Prepare the data for downstream Silver-layer validation and transformation.
- Support repeatable and auditable batch processing.

### Data Flow
GitHub CSV
→ Fabric Master Pipeline
→ Lakehouse Bronze Raw Files
→ This Notebook
→ Downstream Silver Processing

### Design Principle
The original raw file is retained in the Bronze landing zone and is
not modified by this notebook. Processing is performed on data read
from the raw landing zone.


In [25]:
# Pipeline parameters
# These are fallback values for manual notebook testing.
# When executed from the Master Pipeline, these are overridden
# by the Notebook Activity Base parameters.

batch_file_name = "batch_01_history.csv"
load_run_id = "manual-test"

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 27, Finished, Available, Finished, False)

In [26]:
# ============================================================
# Create Bronze Run Log Table
# ============================================================
# This table records the outcome of each Bronze ingestion
# attempt independently from the raw Bronze sales data.
#
# A new record is created for every execution attempt.
# Therefore, retries are visible in the audit history even
# though the actual Bronze data remains idempotent.

run_log_table_name = "bronze_ingestion_log"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {run_log_table_name} (
    run_id STRING,
    batch_file_name STRING,
    ingestion_start_time TIMESTAMP,
    ingestion_end_time TIMESTAMP,
    status STRING,
    rows_received BIGINT,
    rows_written BIGINT,
    failure_reason STRING
)
USING DELTA
""")

print(f"Run log table '{run_log_table_name}' is ready.")

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 28, Finished, Available, Finished, False)

Run log table 'bronze_ingestion_log' is ready.


In [27]:
from datetime import datetime

ingestion_start_time = datetime.utcnow()

print(
    f"Bronze ingestion started for "
    f"'{batch_file_name}' | Run ID: {load_run_id}"
)

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 29, Finished, Available, Finished, False)

Bronze ingestion started for 'batch_03_incremental.csv' | Run ID: run-batch-03


In [28]:
from pyspark.sql.functions import (
    lit,
    current_timestamp
)

# Build the raw Bronze file path from the batch filename parameter.
raw_file_path = f"Files/bronze/raw/{batch_file_name}"


# Read the raw CSV from the Bronze landing zone.
# The source contains quoted text fields with embedded double quotes,
# so the CSV parser is configured explicitly to preserve those fields.
df = (
    spark.read
        .option("header", "true")
        .option("quote", '"')
        .option("escape", '"')
        .csv(raw_file_path)
)


# Add source-file metadata for record-level traceability.
df = df.withColumn(
    "source_file_name",
    lit(batch_file_name)
)


# Add the pipeline execution identifier for audit and traceability.
df = df.withColumn(
    "load_run_id",
    lit(load_run_id)
)


# Record when the batch was processed by the Bronze ingestion notebook.
df = df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

# Inspect a few source records after adding Bronze lineage metadata.
display(
    df.select(
        "Row ID",
        "Order ID",
        "Sales",
        "source_file_name",
        "load_run_id",
        "ingestion_timestamp"
    ).limit(10)
)


# Inspect the inferred data types of the source columns.
df.printSchema()


# Inspect the raw date values before applying any type conversion.
display(
    df.select("Order Date", "Ship Date").limit(10)
)

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3d6a700f-4f7a-49ad-8a40-6608a7acbeda)

root
 |-- Row ID: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: string (nullable = true)
 |-- Order Channel: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- load_run_id: string (nullable = false)
 |-- inges

SynapseWidget(Synapse.DataFrame, 75c790d6-a5b6-4d5d-ada5-6d414259ba2b)

In [29]:
# Check whether the Bronze Delta table already exists.
#
# We perform this check before writing so that development/testing runs
# do not accidentally append the same batch multiple times.

bronze_table_name = "bronze_sales"

table_exists = spark.catalog.tableExists(bronze_table_name)

print(f"Bronze table '{bronze_table_name}' exists: {table_exists}")

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 31, Finished, Available, Finished, False)

Bronze table 'bronze_sales' exists: True


In [30]:
# Normalize Bronze column names to Delta-compatible names.
#
# The raw CSV in Files/bronze/raw/ remains unchanged.
# Only the table representation is normalized so that it can be stored
# reliably as a Delta table and referenced consistently downstream.

import re

def normalize_column_name(column_name):
    """
    Convert source column names into lowercase snake_case names.

    Example:
        'Row ID'       -> 'row_id'
        'Order Date'   -> 'order_date'
        'Postal Code'  -> 'postal_code'
    """
    normalized = re.sub(r"[^A-Za-z0-9]+", "_", column_name)
    normalized = normalized.strip("_").lower()
    return normalized

df_bronze = df.toDF(
    *[normalize_column_name(column) for column in df.columns]
)

print("Bronze column names:")
print(df_bronze.columns)

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 32, Finished, Available, Finished, False)

Bronze column names:
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'order_channel', 'source_file_name', 'load_run_id', 'ingestion_timestamp']


In [31]:
# ============================================================
# Persist Retrieved Batch into Bronze + Record Ingestion Result
# ============================================================
# PURPOSE:
#   Persist the source batch into Bronze and record the outcome
#   of this ingestion attempt.
#
# IDEMPOTENCY:
#   The source file identifies the logical batch. If the same
#   batch is retried, only that batch is replaced in Bronze.
#
# OBSERVABILITY:
#   Every successful or failed Bronze ingestion attempt is
#   recorded in bronze_ingestion_log.
#
# IMPORTANT:
#   The log schema is explicitly defined instead of relying on
#   Spark type inference. This is required because failure_reason
#   can legitimately be NULL for successful executions.
# ============================================================

from datetime import datetime
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

rows_received = df_bronze.count()

# Explicit schema prevents Spark from failing to infer the type
# of failure_reason when its value is None on successful runs.
run_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("batch_file_name", StringType(), False),
    StructField("ingestion_start_time", TimestampType(), False),
    StructField("ingestion_end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_received", LongType(), False),
    StructField("rows_written", LongType(), False),
    StructField("failure_reason", StringType(), True)
])

try:

    # ------------------------------------------------------------
    # Persist the batch into Bronze
    # ------------------------------------------------------------

    if not spark.catalog.tableExists(bronze_table_name):

        # First ingestion: create the Bronze table.
        (
            df_bronze.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(bronze_table_name)
        )

        print(
            f"Created Bronze table and persisted "
            f"{rows_received} rows from '{batch_file_name}'."
        )

    else:

        # Retry/reprocessing:
        # Replace only this source batch.
        #
        # Other batches already stored in Bronze are preserved.
        (
            df_bronze.write
            .format("delta")
            .mode("overwrite")
            .option(
                "replaceWhere",
                f"source_file_name = '{batch_file_name}'"
            )
            .option("mergeSchema", "true")
            .saveAsTable(bronze_table_name)
        )

        print(
            f"Replaced Bronze records for "
            f"'{batch_file_name}' with {rows_received} rows."
        )

    # ------------------------------------------------------------
    # Record successful Bronze ingestion
    # ------------------------------------------------------------

    ingestion_end_time = datetime.utcnow()

    success_log_df = spark.createDataFrame(
        [(
            load_run_id,
            batch_file_name,
            ingestion_start_time,
            ingestion_end_time,
            "SUCCESS",
            rows_received,
            rows_received,
            None
        )],
        schema=run_log_schema
    )

    (
        success_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze_ingestion_log")
    )

    print("Bronze ingestion log recorded: SUCCESS")


except Exception as e:

    # ------------------------------------------------------------
    # Record failed Bronze ingestion
    # ------------------------------------------------------------
    # Capture the actual Bronze/write error and preserve it in
    # the operational log before re-raising the exception.

    ingestion_end_time = datetime.utcnow()
    failure_reason = str(e)

    failed_log_df = spark.createDataFrame(
        [(
            load_run_id,
            batch_file_name,
            ingestion_start_time,
            ingestion_end_time,
            "FAILED",
            rows_received,
            0,
            failure_reason
        )],
        schema=run_log_schema
    )

    (
        failed_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze_ingestion_log")
    )

    print("Bronze ingestion log recorded: FAILED")
    print(f"Failure reason: {failure_reason}")

    # Re-raise the original Bronze exception so Fabric marks
    # the notebook activity as FAILED.
    raise

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 33, Finished, Available, Finished, False)

Replaced Bronze records for 'batch_03_incremental.csv' with 915 rows.
Bronze ingestion log recorded: SUCCESS


In [32]:
# ============================================================
# Check Whether Batch 3 Is Already in Bronze
# ============================================================
# Check before rerunning the append operation so we do not
# accidentally ingest the same batch twice.

from pyspark.sql import functions as F

bronze_batch_count = (
    spark.table(bronze_table_name)
    .filter(F.col("source_file_name") == batch_file_name)
    .filter(F.col("load_run_id") == load_run_id)
    .count()
)

print(f"Batch currently in Bronze: {bronze_batch_count} rows")

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 34, Finished, Available, Finished, False)

Batch currently in Bronze: 915 rows


In [33]:
# Verify the Bronze table after the initial successful ingestion.
#
# This check confirms that:
# 1. The expected number of Batch 1 records was persisted.
# 2. The source-file lineage is present.
# 3. The ingestion run identifier is present.
# 4. The raw business columns are available in the Bronze table.

bronze_df = spark.table("bronze_sales")

print(f"Bronze row count: {bronze_df.count()}")

print("\nBronze schema:")
bronze_df.printSchema()

print("\nSource-file counts:")
display(
    bronze_df
        .groupBy("source_file_name")
        .count()
        .orderBy("source_file_name")
)

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 35, Finished, Available, Finished, False)

Bronze row count: 10974

Bronze schema:
root
 |-- row_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- profit: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- load_run_id: string (nullable = true)
 |-- ingestion_ti

SynapseWidget(Synapse.DataFrame, cbb6559e-b0cf-42a5-98c4-5d7999b36087)

In [34]:
# ============================================================
# Verify Correct CSV Column Alignment
# ============================================================
# Confirm that records previously affected by CSV parsing are
# now correctly aligned in the Bronze layer.

display(
    df_bronze
        .filter(
            df_bronze["row_id"].isin(
                ["33", "106", "256", "300", "340"]
            )
        )
        .select(
            "row_id",
            "order_id",
            "product_id",
            "product_name",
            "sales",
            "quantity",
            "discount",
            "profit"
        )
)

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe00c2b0-9c25-453b-8a91-5b0cd6506586)

In [35]:
# ============================================================
# Diagnostic — Check Bronze Records by Source Batch
# ============================================================

display(
    spark.table("bronze_sales")
    .groupBy("source_file_name")
    .count()
    .orderBy("source_file_name")
)

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f74e2775-932a-4c05-be65-f901b415af52)

In [36]:
##ingestion log

display(
    spark.table("bronze_ingestion_log")
    .orderBy(F.col("ingestion_start_time").desc())
)

StatementMeta(, 76564c87-6116-46b4-8b80-7744cac501b6, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6fd45e8a-71df-487f-b5f9-b7acf309cb34)